#  Twitter Sentiment Model Training
**Pipeline:** Tokenizer → HashingTF → IDF → Logistic Regression


## 1. Imports

In [ ]:
import os
import sys
from pathlib import Path

# Set Hadoop env before any PySpark import (same fix as consumer.py and views.py)
os.environ.setdefault('HADOOP_HOME',           r'C:\hadoop')
os.environ.setdefault('PYSPARK_PYTHON',        sys.executable)
os.environ.setdefault('PYSPARK_DRIVER_PYTHON', sys.executable)
_hadoop_bin = r'C:\hadoop\bin'
if _hadoop_bin not in os.environ.get('PATH', ''):
    os.environ['PATH'] = _hadoop_bin + os.pathsep + os.environ.get('PATH', '')

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace, lower, trim
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, NGram, HashingTF, IDF, StringIndexer, IndexToString
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [2]:
import os
print("HADOOP_HOME       :", os.environ.get("HADOOP_HOME"))
print("winutils.exe found:", os.path.exists(r"C:\hadoop\bin\winutils.exe"))
print("hadoop.dll found  :", os.path.exists(r"C:\hadoop\bin\hadoop.dll"))

HADOOP_HOME       : C:\hadoop
winutils.exe found: True
hadoop.dll found  : True


## 2. Spark Session

In [ ]:
spark = (
    SparkSession.builder
    .appName("TwitterSentimentTraining")
    .config("spark.driver.memory", "4g")
    .config("spark.local.dir",     "C:/tmp/spark")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

## 3. Load & Inspect Data

In [4]:
# Resolve absolute paths to the CSVs (notebook lives in ML-PySpark-Model/notebooks/)
DATASETS_DIR = (Path.cwd().parent / "datasets").resolve()

# PySpark on Windows requires forward slashes
TRAIN_PATH = str(DATASETS_DIR / "twitter_training.csv").replace("\\", "/")
VAL_PATH   = str(DATASETS_DIR / "twitter_validation.csv").replace("\\", "/")

# Fail fast with a clear error if the files aren't where we expect
assert os.path.exists(TRAIN_PATH), f"Training CSV not found at: {TRAIN_PATH}"
assert os.path.exists(VAL_PATH),   f"Validation CSV not found at: {VAL_PATH}"

print("Train CSV:", TRAIN_PATH)
print("Val CSV  :", VAL_PATH)

COLS = ["tweet_id", "entity", "sentiment", "text"]

train_df = spark.read.csv(TRAIN_PATH, header=False, inferSchema=True).toDF(*COLS)
val_df   = spark.read.csv(VAL_PATH,   header=False, inferSchema=True).toDF(*COLS)

# Drop rows with missing text or label
train_df = train_df.dropna(subset=["text", "sentiment"])
val_df   = val_df.dropna(subset=["text", "sentiment"])

print(f"\nTrain rows : {train_df.count()}")
print(f"Val rows   : {val_df.count()}")
print()
train_df.groupBy("sentiment").count().orderBy("count", ascending=False).show()

Train CSV: D:/Twitter_Sentiment_analysis/Real-Time-Twitter-Sentiment-Analysis/ML-PySpark-Model/datasets/twitter_training.csv
Val CSV  : D:/Twitter_Sentiment_analysis/Real-Time-Twitter-Sentiment-Analysis/ML-PySpark-Model/datasets/twitter_validation.csv

Train rows : 73996
Val rows   : 1000

+----------+-----+
| sentiment|count|
+----------+-----+
|  Negative|22358|
|  Positive|20655|
|   Neutral|18108|
|Irrelevant|12875|
+----------+-----+



## 4. Text Cleaning
Strips URLs, @mentions, and non-alphabetic characters. Lowercases everything.

In [5]:
def clean_text(df):
    return (
        df
        .withColumn("text", regexp_replace(col("text"), r"http\S+", ""))
        .withColumn("text", regexp_replace(col("text"), r"@\w+", ""))
        .withColumn("text", regexp_replace(col("text"), r"[^a-zA-Z\s]", ""))
        .withColumn("text", trim(lower(col("text"))))
    )

train_df = clean_text(train_df)
val_df   = clean_text(val_df)

train_df.select("sentiment", "text").show(5, truncate=90)

+---------+-------------------------------------------------------+
|sentiment|                                                   text|
+---------+-------------------------------------------------------+
| Positive|    im getting on borderlands and i will murder you all|
| Positive|     i am coming to the borders and i will kill you all|
| Positive|      im getting on borderlands and i will kill you all|
| Positive|     im coming on borderlands and i will murder you all|
| Positive|im getting on borderlands  and i will murder you me all|
+---------+-------------------------------------------------------+
only showing top 5 rows



## 5. Build ML Pipeline
Each stage transforms the data and passes it to the next:
- **StringIndexer** — converts sentiment string to numeric label
- **Tokenizer** — splits text into word list
- **NGram (n=2)** — creates bigrams (e.g. "absolutely loved") alongside unigrams
- **HashingTF** — maps word/bigram list to fixed-size feature vector
- **IDF** — down-weights words that appear in many tweets (like 'the', 'a')
- **LogisticRegression** — multi-class classifier on the TF-IDF features

In [ ]:
label_indexer = StringIndexer(
    inputCol="sentiment", outputCol="label", handleInvalid="skip"
)

tokenizer = Tokenizer(inputCol="text", outputCol="words")

ngram = NGram(n=2, inputCol="words", outputCol="bigrams")

hashing_tf = HashingTF(
    inputCol="bigrams", outputCol="raw_features", numFeatures=20000
)

idf = IDF(
    inputCol="raw_features", outputCol="features", minDocFreq=5
)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=20,
    regParam=0.01
)

pipeline = Pipeline(stages=[label_indexer, tokenizer, ngram, hashing_tf, idf, lr])
print("Pipeline stages:", [s.__class__.__name__ for s in pipeline.getStages()])

## 6. Train
Fits all pipeline stages on the training data. Takes 2–4 minutes.

In [7]:
print("Training started...")
model = pipeline.fit(train_df)
print("Training complete.")

# Label mapping produced by StringIndexer
labels = model.stages[0].labels
print("Label mapping:", {i: lbl for i, lbl in enumerate(labels)})

Training started...
Training complete.
Label mapping: {0: 'Negative', 1: 'Positive', 2: 'Neutral', 3: 'Irrelevant'}


## 7. Evaluate on Validation Set

In [8]:
predictions = model.transform(val_df)

# Map numeric predictions back to label names for readability
index_to_label = IndexToString(
    inputCol="prediction", outputCol="predicted_label",
    labels=model.stages[0].labels
)
predictions = index_to_label.transform(predictions)

evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = evaluator.evaluate(predictions)
print(f"Validation Accuracy: {accuracy * 100:.2f}%")

print("\nConfusion Matrix (actual vs predicted):")
predictions.groupBy("sentiment", "predicted_label") \
           .count() \
           .orderBy("sentiment", "predicted_label") \
           .show(20)

Validation Accuracy: 82.20%

Confusion Matrix (actual vs predicted):
+----------+---------------+-----+
| sentiment|predicted_label|count|
+----------+---------------+-----+
|Irrelevant|     Irrelevant|  131|
|Irrelevant|       Negative|   14|
|Irrelevant|        Neutral|    5|
|Irrelevant|       Positive|   22|
|  Negative|     Irrelevant|    5|
|  Negative|       Negative|  236|
|  Negative|        Neutral|    6|
|  Negative|       Positive|   19|
|   Neutral|     Irrelevant|    9|
|   Neutral|       Negative|   27|
|   Neutral|        Neutral|  217|
|   Neutral|       Positive|   32|
|  Positive|     Irrelevant|    8|
|  Positive|       Negative|   13|
|  Positive|        Neutral|   18|
|  Positive|       Positive|  238|
+----------+---------------+-----+



## 8. Save Model
Saves the entire fitted pipeline (not just the LR weights) so the consumer can apply identical preprocessing at inference time.

In [9]:
# Use absolute path here too, for the same reason as cell 6
MODEL_PATH = str((Path.cwd().parent / "saved_models" / "spark_lr_pipeline").resolve()).replace("\\", "/")

model.write().overwrite().save(MODEL_PATH)
print(f"Model saved → {MODEL_PATH}")

Model saved → D:/Twitter_Sentiment_analysis/Real-Time-Twitter-Sentiment-Analysis/ML-PySpark-Model/saved_models/spark_lr_pipeline
